# Unified VAE Lipreading Notebook

This notebook merges the workflows from your existing notebooks into one place:
- VAE training from raw video/align data + latent export
- MLP training from exported latent tensors
- DNN + XGBoost experiments from latent tensors
- MLP training from precomputed vector features (.npy)

Use the `RUN_MODE` setting in the config cell to pick the workflow you want.

### How this notebook is organized
Each section below is one stage of the lipreading pipeline. The idea is to keep the original training code, the GPU-friendly latent workflows, and the vector-feature experiments in one notebook so you can switch modes instead of maintaining separate copies.

In [ ]:
# Core imports
import os
import random
from dataclasses import dataclass
from typing import List, Tuple, Dict

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import torchvision
import torchvision.transforms as T
from torchvision.models.video import R3D_18_Weights

import matplotlib.pyplot as plt
from tqdm import tqdm

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("xgboost import failed:", e)

In [ ]:
# Configuration
@dataclass
class Config:
    # Select one:
    #   vae_train_export
    #   mlp_from_latent
    #   dnn_xgb_from_latent
    #   mlp_from_vector
    RUN_MODE: str = "mlp_from_latent"

    # General
    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Raw dataset layout for VAE mode
    # Expected pattern under DATA_ROOT:
    # subset1/s1 + subset1/align, ... subset10/s10 + subset10/align
    DATA_ROOT: str = "data"
    NUM_SUBSETS: int = 10

    # VAE params
    LATENT_DIM: int = 128
    NUM_CLASSES: int = 53
    VAE_EPOCHS: int = 10
    VAE_LR: float = 1e-4
    KL_WEIGHT: float = 0.1

    # Latent MLP params
    USE_MU_ONLY: bool = False
    MLP_EPOCHS: int = 250
    MLP_LR: float = 1e-3
    MLP_BATCH_SIZE: int = 64

    # DNN/XGB params
    LATENT_AUG_SAMPLES: int = 5

    # Vector feature mode files
    TRAIN_FEATURES_NPY: str = "train_feats_full.npy"
    TEST_FEATURES_NPY: str = "test_feats_full.npy"

    # Label files
    TR_LABELS: str = "tr_labels.pt"
    VAL_LABELS: str = "val_labels.pt"
    TEST_LABELS: str = "test_labels.pt"

cfg = Config()
print(cfg)
print("Using device:", cfg.DEVICE)

### Why this configuration cell matters
This is the control panel for the whole notebook. `RUN_MODE` decides which workflow runs, and the path / hyperparameter settings let the same notebook work both for the original VAE pipeline and the downstream GPU-only classifier experiments.

In [ ]:
# Reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)

### Why we seed everything here
The notebooks were split across runs and machines, so this keeps the random number generators aligned. That makes the train/val/test splits and model results easier to reproduce when you rerun the merged notebook later.

## Shared Helpers (labels, tensors, metrics)

In [ ]:
def build_label_index(labels: List[str]) -> Tuple[List[str], Dict[str, int], np.ndarray]:
    unique_labels = sorted(set(labels))
    label_to_idx = {lbl: idx for idx, lbl in enumerate(unique_labels)}
    labels_idx = np.array([label_to_idx[l] for l in labels], dtype=np.int64)
    return unique_labels, label_to_idx, labels_idx


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int = 64, shuffle: bool = False) -> DataLoader:
    X = X.astype(np.float32)
    y = y.astype(np.int64)
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def evaluate_loader(model: nn.Module, dl: DataLoader, device: str = "cpu"):
    model.eval()
    with torch.no_grad():
        logits = torch.cat([model(xb.to(device)) for xb, _ in dl], dim=0)
        targets = torch.cat([yb for _, yb in dl], dim=0).to(device)
        preds = logits.argmax(dim=1)
    acc = accuracy_score(targets.cpu().numpy(), preds.cpu().numpy())
    return acc, targets.cpu().numpy(), preds.cpu().numpy()

### Shared helpers
These helpers are reused across every mode. They turn text labels into numeric classes, build PyTorch loaders, and provide a common evaluation path so the notebook does not duplicate the same data-handling logic in every workflow.

## VAE Workflow (raw video -> vae_trained.pt + tr_/val_/test_ latent files)

In [ ]:
def collect_video_align_pairs(data_root: str = "data", num_subsets: int = 10):
    pairs = []
    for subset in range(1, num_subsets + 1):
        subset_path = f"subset{subset}"
        video_dir = os.path.join(data_root, subset_path, f"s{subset}")
        align_dir = os.path.join(data_root, subset_path, "align")

        if not os.path.isdir(video_dir) or not os.path.isdir(align_dir):
            continue

        video_files = [f for f in os.listdir(video_dir) if f.endswith(".mpg")]
        for vf in video_files:
            stem = os.path.splitext(vf)[0]
            align_path = os.path.join(align_dir, stem + ".align")
            video_path = os.path.join(video_dir, vf)
            if os.path.exists(align_path):
                pairs.append((video_path, align_path))

    print(f"Found {len(pairs)} video-align pairs")
    return pairs


def split_pairs(pairs):
    train_pairs, valtest_pairs = train_test_split(pairs, test_size=0.4, random_state=42)
    val_pairs, test_pairs = train_test_split(valtest_pairs, test_size=0.5, random_state=42)
    print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}")
    return train_pairs, val_pairs, test_pairs


_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((112, 112)),
    T.ToTensor(),
    T.Normalize([0.43216, 0.394666, 0.37645], [0.22803, 0.22145, 0.216989]),
])


def parse_align_file(align_path: str):
    alignments = []
    with open(align_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                start, end, word = parts
                alignments.append((int(start), int(end), word))
    return alignments


def extract_word_frames(video_path: str, alignments):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    word_frames = []
    for start, end, word in alignments:
        start_idx = int(start / 1000)
        end_idx = int(end / 1000)
        seq = frames[start_idx:end_idx]
        if len(seq) > 0:
            word_frames.append((word, seq))
    return word_frames


def preprocess_frames(frames):
    processed = [_transform(frame) for frame in frames]
    video_tensor = torch.stack(processed)
    video_tensor = video_tensor.permute(1, 0, 2, 3)
    return video_tensor.unsqueeze(0)

### Raw video preprocessing for the VAE
This block is the bridge from the dataset layout to model input. It finds matching video/alignment pairs, splits them into train/validation/test sets, parses the `.align` files, and turns each word segment into a normalized tensor the VAE can train on.

In [ ]:
class VideoVAE(nn.Module):
    def __init__(self, latent_dim=128, num_classes=53):
        super().__init__()
        base = torchvision.models.video.r3d_18(weights=R3D_18_Weights.DEFAULT)
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.encoder_out_dim = 512

        self.fc_mu = nn.Linear(self.encoder_out_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.encoder_out_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, self.encoder_out_dim),
            nn.ReLU(),
            nn.Linear(self.encoder_out_dim, self.encoder_out_dim),
        )

        self.classifier = nn.Linear(latent_dim, num_classes)

    def encode(self, x):
        x = self.encoder(x)
        x = x.flatten(1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        logits = self.classifier(mu)
        return recon, mu, logvar, logits

### VAE model definition
This model reuses a pretrained 3D ResNet as the encoder, then learns a latent representation for reconstruction and classification. Keeping it here makes the original training code available without needing a separate notebook.

In [ ]:
def train_vae_and_export(cfg: Config):
    pairs = collect_video_align_pairs(cfg.DATA_ROOT, cfg.NUM_SUBSETS)
    if len(pairs) == 0:
        raise ValueError("No video-align pairs found. Check DATA_ROOT and subset structure.")

    train_pairs, val_pairs, test_pairs = split_pairs(pairs)

    all_words = []
    for _, align_path in train_pairs:
        for _, _, word in parse_align_file(align_path):
            all_words.append(word)
    label_encoder = {w: i for i, w in enumerate(sorted(set(all_words)))}

    vae = VideoVAE(latent_dim=cfg.LATENT_DIM, num_classes=cfg.NUM_CLASSES).to(cfg.DEVICE)
    optimizer = optim.Adam(vae.parameters(), lr=cfg.VAE_LR)
    recon_loss_fn = nn.MSELoss()
    cls_loss_fn = nn.CrossEntropyLoss()

    vae.train()
    for epoch in range(cfg.VAE_EPOCHS):
        running_loss = 0.0
        count = 0

        for video_path, align_path in tqdm(train_pairs, desc=f"VAE epoch {epoch+1}/{cfg.VAE_EPOCHS}"):
            alignments = parse_align_file(align_path)
            word_frames = extract_word_frames(video_path, alignments)

            for word, frames in word_frames:
                if word not in label_encoder:
                    continue

                x = preprocess_frames(frames).to(cfg.DEVICE)
                y = torch.tensor([label_encoder[word]], device=cfg.DEVICE)

                optimizer.zero_grad()
                recon, mu, logvar, logits = vae(x)

                target = vae.encoder(x).flatten(1).detach()
                recon_loss = recon_loss_fn(recon, target)
                kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.size(0)
                cls_loss = cls_loss_fn(logits, y)

                loss = recon_loss + cfg.KL_WEIGHT * kl_loss + cls_loss
                loss.backward()
                optimizer.step()

                running_loss += float(loss.item())
                count += 1

        avg_loss = running_loss / max(count, 1)
        print(f"Epoch {epoch+1}: loss={avg_loss:.4f}")

    torch.save(vae.state_dict(), "vae_trained.pt")
    print("Saved vae_trained.pt")

    def export_latents(split_pairs, prefix):
        vae.eval()
        all_mu, all_logvar, all_labels = [], [], []

        with torch.no_grad():
            for video_path, align_path in tqdm(split_pairs, desc=f"Export {prefix}"):
                alignments = parse_align_file(align_path)
                word_frames = extract_word_frames(video_path, alignments)
                for word, frames in word_frames:
                    x = preprocess_frames(frames).to(cfg.DEVICE)
                    mu, logvar = vae.encode(x)
                    all_mu.append(mu.cpu())
                    all_logvar.append(logvar.cpu())
                    all_labels.append(word)

        torch.save(all_mu, f"{prefix}_mu.pt")
        torch.save(all_logvar, f"{prefix}_logvar.pt")
        torch.save(all_labels, f"{prefix}_labels.pt")
        print(f"Saved {prefix}_mu.pt, {prefix}_logvar.pt, {prefix}_labels.pt")

    export_latents(train_pairs, "tr")
    export_latents(val_pairs, "val")
    export_latents(test_pairs, "test")

### VAE training and latent export
This is the expensive stage that actually fits the VAE, saves the trained weights, and exports the latent statistics for later classifiers. It stays separate because the downstream classifiers depend on these saved latent files instead of retraining the VAE each time.

## Latent Feature Loaders

In [ ]:
def load_latent_triplet(prefix: str = "tr", use_mu_only: bool = False):
    logvar = torch.load(f"{prefix}_logvar.pt")
    mu = torch.load(f"{prefix}_mu.pt")
    labels = torch.load(f"{prefix}_labels.pt")

    features = []
    for i in range(len(mu)):
        m = mu[i][0].detach().numpy()
        if use_mu_only:
            features.append(m)
        else:
            l = logvar[i][0].detach().numpy()
            features.append(np.concatenate((l, m), axis=0))

    X = np.array(features, dtype=np.float32)
    return X, labels


def load_train_val_test_from_latents(use_mu_only: bool = False):
    X_train, y_train_text = load_latent_triplet("tr", use_mu_only=use_mu_only)
    X_val, y_val_text = load_latent_triplet("val", use_mu_only=use_mu_only)
    X_test, y_test_text = load_latent_triplet("test", use_mu_only=use_mu_only)

    all_labels_text = list(y_train_text) + list(y_val_text) + list(y_test_text)
    unique_labels, label_to_idx, _ = build_label_index(all_labels_text)

    y_train = np.array([label_to_idx[t] for t in y_train_text], dtype=np.int64)
    y_val = np.array([label_to_idx[t] for t in y_val_text], dtype=np.int64)
    y_test = np.array([label_to_idx[t] for t in y_test_text], dtype=np.int64)

    print("Latent shapes:", X_train.shape, X_val.shape, X_test.shape)
    print("Num classes:", len(unique_labels))

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), unique_labels, label_to_idx

### Latent feature loaders
These helpers read the saved `mu`, `logvar`, and label files from the VAE stage and turn them into classifier-ready arrays. This keeps the downstream notebooks lightweight because they can train directly from exported features instead of raw video.

## MLP from Latents

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, d_out),
        )

    def forward(self, x):
        return self.net(x)


def train_mlp(train_dl, valid_dl, d_in, d_out, cfg: Config):
    model = MLPClassifier(d_in=d_in, d_out=d_out).to(cfg.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.MLP_LR, weight_decay=1e-4)

    for epoch in range(1, cfg.MLP_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(cfg.DEVICE), yb.to(cfg.DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item()) * xb.size(0)

        if epoch % 10 == 0:
            val_acc, _, _ = evaluate_loader(model, valid_dl, device=cfg.DEVICE)
            print(f"Epoch {epoch:03d} | train loss={epoch_loss/len(train_dl.dataset):.4f} | val acc={val_acc*100:.2f}%")

    return model


def run_mlp_from_latent(cfg: Config):
    (X_train, y_train), (X_val, y_val), (X_test, y_test), unique_labels, label_to_idx = load_train_val_test_from_latents(
        use_mu_only=cfg.USE_MU_ONLY
    )

    train_dl = make_loader(X_train, y_train, batch_size=cfg.MLP_BATCH_SIZE, shuffle=True)
    val_dl = make_loader(X_val, y_val, batch_size=cfg.MLP_BATCH_SIZE, shuffle=False)
    test_dl = make_loader(X_test, y_test, batch_size=cfg.MLP_BATCH_SIZE, shuffle=False)

    model = train_mlp(train_dl, val_dl, d_in=X_train.shape[1], d_out=len(unique_labels), cfg=cfg)

    test_acc, y_true, y_pred = evaluate_loader(model, test_dl, device=cfg.DEVICE)
    print(f"TEST accuracy: {test_acc*100:.2f}%")
    print(classification_report(y_true, y_pred, digits=4))

    out_name = "MLPClassifier_mu_only.pt" if cfg.USE_MU_ONLY else "MLPClassifier_full.pt"
    torch.save(model, out_name)
    print(f"Saved {out_name}")

    if "sil" in label_to_idx:
        sil_idx = label_to_idx["sil"]
        sil_guess = int(np.sum(y_pred == sil_idx))
        sil_correct = int(np.sum((y_pred == sil_idx) & (y_true == sil_idx)))
        total_sil_true = int(np.sum(y_true == sil_idx))
        print(f"sil guessed: {sil_guess}, sil correctly guessed: {sil_correct}, sil true count: {total_sil_true}")

### MLP classifier for latent features
This block defines the small feed-forward classifier used on top of the exported VAE features. It is intentionally simpler than the VAE because the goal here is to test how much class information is already present in the latent space.

## DNN + XGBoost from Latents

In [ ]:
class SimpleDNN(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def sample_latent_aug(mu_list, logvar_list, labels_idx, k=5):
    mu = torch.stack(mu_list).view(len(mu_list), -1)
    logvar = torch.stack(logvar_list).view(len(logvar_list), -1)
    std = (0.5 * logvar).exp()

    N, D = mu.shape
    eps = torch.randn(N, k, D)
    z = mu.unsqueeze(1) + eps * std.unsqueeze(1)
    z = z.reshape(-1, D).numpy().astype(np.float32)

    y = np.array(labels_idx, dtype=np.int64)
    y = np.repeat(y, k)
    return z, y


def run_dnn_xgb_from_latent(cfg: Config):
    tr_mu = torch.load("tr_mu.pt")
    tr_logvar = torch.load("tr_logvar.pt")
    tr_labels = torch.load("tr_labels.pt")

    val_mu = torch.load("val_mu.pt")
    val_logvar = torch.load("val_logvar.pt")
    val_labels = torch.load("val_labels.pt")

    all_labels = list(tr_labels) + list(val_labels)
    unique_labels, label_to_idx, _ = build_label_index(all_labels)
    y_tr_idx = [label_to_idx[x] for x in tr_labels]
    y_val_idx = [label_to_idx[x] for x in val_labels]

    X_train, y_train = sample_latent_aug(tr_mu, tr_logvar, y_tr_idx, k=cfg.LATENT_AUG_SAMPLES)
    X_val, y_val = sample_latent_aug(val_mu, val_logvar, y_val_idx, k=cfg.LATENT_AUG_SAMPLES)

    print("DNN/XGB features:", X_train.shape, X_val.shape)

    # DNN baseline
    dnn = SimpleDNN(input_dim=X_train.shape[1], num_classes=len(unique_labels)).to(cfg.DEVICE)
    opt = torch.optim.AdamW(dnn.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()

    train_dl = make_loader(X_train, y_train, batch_size=256, shuffle=True)
    val_dl = make_loader(X_val, y_val, batch_size=256, shuffle=False)

    for epoch in range(1, 41):
        dnn.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(cfg.DEVICE), yb.to(cfg.DEVICE)
            opt.zero_grad()
            logits = dnn(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

        if epoch % 10 == 0:
            val_acc, _, _ = evaluate_loader(dnn, val_dl, device=cfg.DEVICE)
            print(f"DNN epoch {epoch:02d} | val acc={val_acc:.4f}")

    if XGBOOST_AVAILABLE:
        clf = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.05,
            objective="multi:softmax",
            num_class=len(unique_labels),
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=cfg.SEED,
        )
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_val)
        acc = accuracy_score(y_val, y_pred)
        print(f"XGBoost validation accuracy: {acc:.4f}")
    else:
        print("Skipping XGBoost because xgboost is not available.")

### DNN and XGBoost on augmented latents
This experiment re-samples the latent vectors to create more training examples, then compares a neural network baseline with XGBoost. It exists to check whether a stronger tabular classifier can beat the simple MLP on the same learned features.

## MLP from Precomputed Vector Features

In [ ]:
def run_mlp_from_vector(cfg: Config):
    X = np.load(cfg.TRAIN_FEATURES_NPY).astype(np.float32)
    labels_text = torch.load(cfg.TR_LABELS)

    unique_labels, label_to_idx, y = build_label_index(labels_text)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=cfg.SEED
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=cfg.SEED
    )

    train_dl = make_loader(X_train, y_train, batch_size=cfg.MLP_BATCH_SIZE, shuffle=True)
    val_dl = make_loader(X_val, y_val, batch_size=cfg.MLP_BATCH_SIZE, shuffle=False)
    test_dl = make_loader(X_test, y_test, batch_size=cfg.MLP_BATCH_SIZE, shuffle=False)

    model = train_mlp(train_dl, val_dl, d_in=X.shape[1], d_out=len(unique_labels), cfg=cfg)

    test_acc, y_true, y_pred = evaluate_loader(model, test_dl, device=cfg.DEVICE)
    print(f"TEST accuracy (vector split): {test_acc*100:.2f}%")
    print(classification_report(y_true, y_pred, digits=4))

    # Optional external test set if provided
    if os.path.exists(cfg.TEST_FEATURES_NPY) and os.path.exists(cfg.TEST_LABELS):
        X_ext = np.load(cfg.TEST_FEATURES_NPY).astype(np.float32)
        y_ext_text = torch.load(cfg.TEST_LABELS)
        y_ext = np.array([label_to_idx[t] for t in y_ext_text if t in label_to_idx], dtype=np.int64)

        # Keep feature rows aligned with mapped labels
        valid_mask = np.array([t in label_to_idx for t in y_ext_text], dtype=bool)
        X_ext = X_ext[valid_mask]

        ext_dl = make_loader(X_ext, y_ext, batch_size=cfg.MLP_BATCH_SIZE, shuffle=False)
        ext_acc, _, _ = evaluate_loader(model, ext_dl, device=cfg.DEVICE)
        print(f"External test accuracy (vector files): {ext_acc*100:.2f}%")

    torch.save(model, "MLPClassifier_vector.pt")
    print("Saved MLPClassifier_vector.pt")

### MLP on precomputed vector features
This branch is for the version of the data that already lives in `.npy` feature files. It lets you skip the VAE entirely and train the same classifier architecture on ready-made vectors when that is the faster or more practical option.

## Run Dispatcher

In [ ]:
if cfg.RUN_MODE == "vae_train_export":
    train_vae_and_export(cfg)

elif cfg.RUN_MODE == "mlp_from_latent":
    run_mlp_from_latent(cfg)

elif cfg.RUN_MODE == "dnn_xgb_from_latent":
    run_dnn_xgb_from_latent(cfg)

elif cfg.RUN_MODE == "mlp_from_vector":
    run_mlp_from_vector(cfg)

else:
    raise ValueError(f"Unknown RUN_MODE: {cfg.RUN_MODE}")

### Run dispatcher
This final cell is the entry point that chooses which workflow to execute. It keeps the notebook merged while still letting you run only the stage you need on a given machine.

## Notes
- If you are only training classifiers on Sam's GPU, set `RUN_MODE` to `mlp_from_latent`, `dnn_xgb_from_latent`, or `mlp_from_vector`.
- If latent files are not generated yet, run `vae_train_export` first.
- You can keep one notebook and just switch the mode/config per machine.